# 4 · How realistic is this market?

Realism here is a set of measurements against real markets, with what the
model still gets wrong written down beside them. There is no single score.
Two checks describe the shipped default:

- **The one-year panel.** Statistics of daily returns, volume and
  correlation, the index's yearly return and how the VIX reacts to a bad
  day, each measured on thirty seeds of one simulated year (360 for the
  index's return and its rate of 3% falls) and compared with the range
  real markets have produced.
- **The long-run check.** 270 simulated histories of twenty-one years
  each, plus replays of 2008 and 2020, the real 2020-21 and 2022 macro
  paths and the packaged recession, read against the 190 rows that count
  toward pt-v22's grade. Among them are the forty criteria a user would
  notice first: how violent crashes are, how often fear takes hold, how
  many bear markets a decade brings, what the index returns over the long
  run, whether a rule that reads only prices, published fundamentals or
  published macro data finds an edge real markets do not offer, what size
  costs in the book, how bonds move beside stocks, and how a market
  recovers.

The first cell confirms that the preset you are running is the one the
tables below describe.

In [1]:
import math

import tradefloor as tf
from tradefloor import envelope as env
from tradefloor import facts
from tradefloor.facts import LABELS

running = tf.model_preset()["name"]
print("preset you are running      :", running)
print("preset the envelope describes:", env.PRESET)
print("certified horizon           :", env.CERTIFIED_HORIZON_DAYS, "trading days")
assert running == env.PRESET, "the tables below describe another preset"

preset you are running      : pt-v22
preset the envelope describes: pt-v22
certified horizon           : 252 trading days


## The one-year panel

Measured on forty instruments over 252 trading days, at thirty seeds, and at
360 for the index's yearly return and how often it falls 3% or more. The
rows come in four groups:

- **shape**: fourteen statistics of how prices, volume and correlation
  behave, from `envelope.CERTIFIED`
- **level**: the index's yearly return, from `envelope.CERTIFIED_LEVEL`
- **crisis**: how far the VIX jumps on a day the index falls 1% or 3%, and
  how often the index falls 3% or more, from `envelope.CERTIFIED_CRISIS`
- **dispersion**: how unevenly sectors suffer when the VIX is above 30

The level and crisis rows are measured on a protocol where the roster is
drawn afresh with each seed, because a level that describes the model
cannot be read off one roster, so they sit in their own tables.

`envelope.score` grades a panel against the band real markets produce for
each row.

In [2]:
panel = {**env.certified_panel(), **env.CERTIFIED_LEVEL, **env.CERTIFIED_CRISIS}
year = env.score(panel, horizon_days=252)

print(f"ruler: {year['ruler']}\n")
print(f"{'row':26s} {'group':>10s} {'measured':>9s} {'real band':>15s}")
for name, row in year["statistics"].items():
    lo, hi = row["band"]
    print(f"{LABELS.get(name, name):26s} {row['group']:>10s} "
          f"{row['measured']:9.4f} {f'{lo:g} to {hi:g}':>15s}  "
          f"{'in' if row['in_band'] else 'OUT'}")

print(f"\n{year['in_band']} of {year['of']} in band: "
      f"shape {year['shape_in_band']}/{year['shape_of']}, "
      f"level {year['level_in_band']}/{year['level_of']}, "
      f"crisis {year['crisis_in_band']}/{year['crisis_of']}, "
      f"dispersion {year['dispersion_in_band']}/{year['dispersion_of']}")

ruler: facts.REAL_MARKETS_RULED

row                             group  measured       real band
annualised vol %                shape   19.9402        12 to 41  in
excess kurtosis                 shape    6.7198       -13 to 24  in
return acf(1)                   shape   -0.0049   -0.07 to 0.06  in
|return| acf(1)                 shape    0.1052    0.02 to 0.17  in
|return| acf(5)                 shape    0.0281    -0.03 to 0.1  in
|return| acf(20)                shape    0.0044   -0.05 to 0.06  in
cross-sectional corr            shape    0.3115    0.09 to 0.49  in
volume vs |return|              shape    0.5928    0.35 to 0.64  in
leverage, r vs |r+1|            shape   -0.0434      -0.11 to 0  in
volume change acf(1)            shape   -0.2414    -0.3 to -0.2  in
corr, down vs up days           shape    0.0646   -0.15 to 0.23  in
corr, after down vs up          shape    0.0734   -0.15 to 0.33  in
same-sector excess corr         shape    0.1264    0.04 to 0.23  in
corr persistence ac

All nineteen rows are inside their bands at one year. Two of them are read
over 360 seeds rather than thirty: how often the index falls 3% or more in
a session, which is rare by construction, one or two sessions a year, and
the index's yearly return. A few seeds decide each of them, so thirty
one-year runs read them loosely. The 3% row is the falls on every seed over the sessions of every seed, 1.25%, and the return is the mean across seeds.
The long-run check below reads both the same way on the same seeds.

The bands are wide on purpose. Each one is the range real markets have
produced across many one-year windows, back to 1987 for most rows and to
1928 for the index's return, so a row in band reads like some real year.
That is a weaker claim than reading like the average year.

`room_sd` says how far inside its band a row sits, in units of its own
spread across seeds. A row barely inside is one unlucky seed from
outside, and a plain in-or-out check cannot tell the two apart.

In [3]:
rows = [(k, r["room_sd"]) for k, r in year["statistics"].items()
        if r["in_band"] and r["room_sd"] is not None]
for k, room in sorted(rows, key=lambda kv: kv[1]):
    flag = "  <- thin" if room < 0.5 else ""
    print(f"  {LABELS.get(k, k):26s} {room:6.2f} sd inside{flag}")

  index drift %/yr             0.31 sd inside  <- thin
  leverage, r vs |r+1|         0.56 sd inside
  |return| acf(1)              0.68 sd inside
  corr, down vs up days        1.01 sd inside
  |return| acf(5)              1.02 sd inside
  |return| acf(20)             1.16 sd inside
  annualised vol %             1.23 sd inside
  return acf(1)                1.23 sd inside
  cross-sectional corr         1.65 sd inside
  corr persistence acf(1)      1.90 sd inside
  corr, after down vs up       1.93 sd inside
  volume vs |return|           3.30 sd inside
  volume change acf(1)         3.47 sd inside
  same-sector excess corr     13.51 sd inside
  excess kurtosis             14.67 sd inside


One row sits within half a seed standard deviation of an edge: the index's
yearly return, 0.31 sd inside. The index drifts 7.33% a year on this
protocol, inside a band of 1.1 to 10.3, but its spread
across seeds is wide, so any single run can read it outside its band. The
leverage effect is next, at 0.56 sd.

## The 2015-2025 ruler

The bands above are `envelope.score`'s default. The library also keeps an
older set cut from 2015-2025 alone, `facts.REAL_MARKETS`, which is
narrower on some rows because one decade varies less than four. Scoring
the same numbers against it shows what a user comparing the model with
the recent past would see.

In [4]:
decade = env.score(panel, horizon_days=252, basis="shipped")
print(f"{decade['ruler']}: {decade['in_band']} of {decade['of']} in band")
for name, row in decade["statistics"].items():
    if row["in_band"] is False:
        lo, hi = row["band"]
        print(f"  out: {name} {row['measured']:.4f}, band {lo:g} to {hi:g}")
print(f"  no band on this ruler: {', '.join(decade['unreadable'])}")

facts.REAL_MARKETS: 18 of 18 in band
  no band on this ruler: crisis_sector_dispersion


Every row is in on this ruler, the 3% fall row included: its 2015-2025
band, 0.47 to 1.96, reaches lower than the long-record one. On pt-v19 one
row was out here: `sector_excess_corr`, how much more two stocks in the
same sector move together than two picked at random, at 0.090 against a
floor of 0.11. pt-v22 reads 0.126.

## The two-year ruler

A statistic read over two years is a different measurement from the same
statistic read over one, so it needs bands derived at two years.
`envelope.score` picks the ruler from `horizon_days`. Below is the
two-year panel, measured at 504 days, against the two-year bands.

In [5]:
two_years = {k: v for k, v in env.MEASURED_504.items() if v is not None}
for basis in ("ruled", "shipped"):
    s = env.score(two_years, horizon_days=504, basis=basis)
    out = [k for k, r in s["statistics"].items() if r["in_band"] is False]
    print(f"{s['ruler']}: {s['in_band']} of {s['of']} in band")
    print(f"    out: {', '.join(out) or 'none'};  "
          f"no band: {', '.join(s['unreadable']) or 'none'}")

facts.REAL_MARKETS_RULED_504: 15 of 15 in band
    out: none;  no band: none
envelope.BANDS_504: 13 of 14 in band
    out: excess_kurtosis;  no band: crisis_sector_dispersion


At two years every row is in on the long-record bands. On the
2015-2025 bands one is out: `excess_kurtosis`, how fat the tails of daily
returns are, reads 6.84 against a floor of 7.1. The envelope certifies one
year only,
because that is where the thirty-seed certification was run, and `check`
below refuses a question beyond it.

## The long-run check

New in 0.8.0. The one-year panel cannot see what a long backtest would:
whether crashes are as violent as real ones, how often fear takes hold,
how many bear markets a decade brings, and what the index returns over
twenty years. For pt-v22 the long-run check counts 190 rows toward the
grade and reports 41 more beside them. Among the 190 are the forty
criteria pt-v20 and pt-v21 were graded on, all but R6, which pt-v22
reports without counting. Most of the others were registered for pt-v22,
the futures, options and margin among them. The forty are the fifteen
adopted on 2026-09-23, C4a and C4b added on 2026-09-24, eleven
registered for pt-v20 the same day (B9, C5 to C9, R1 to R4 and E1), D2 on
2026-09-24, F1, L1, R5, R6 and C10 on 2026-09-25, and R7a, R7b, S1a, S1b,
S2 and V1 on 2026-09-26. Each has a tolerance a
person can read, set by asking whether a gap would mislead a user rather
than by asking for an exact match.

The A rows replay 2008 and 2020 with the real VIX imposed, so they test
how the market responds to fear rather than whether it produces the fear
itself. The B rows let the market run free for twenty-one years. The C
rows look for illusions a trading bot could learn, and D1 is the one-year
panel above. C4a and C4b ask whether 65-minute returns reverse more than
real ones, and whether simple reversal and momentum rules beat
buy-and-hold on the published suite of 20 markets. C5 to C8 ask the same
of a name's own variance, a value screen, 12-1 momentum and a one-day
reversal book, and C9 what size costs in the agent-facing book. B9 reads
how much the index's annual return varies, R1 to R4 how Treasuries and
corporate bonds move against real yields and stocks, and E1 how far
aggregate earnings fall in a contraction. D2, F1 and L1 drive the real
2020-21 macro path and read the fall, its speed, the recovery, and
whether prices turn before earnings; R5 and R6 drive 2022 and read the
fall and how far the market's P/E moves per point of the corporate
yield. C10, R7a and R7b ask whether published macro data or a published
rate decision can be traded for an edge. S1a, S1b and S2 ask whether the
packaged recession recovers, and V1 whether the index's variance over two
and five years stays in proportion to one year's.

For pt-v22 the long-run rows are read on 270 histories of 21 years, and
graded at the twenty-third registration. The grade, its scripts and its
inputs are published under `validation/pt-v22/`.
The result is part of the preset's measured record, which ships in the
wheel.

In [6]:
record = tf.preset_record(env.PRESET)
long_run = record["long_run"]
measured = long_run["measured"]
print(f"{env.PRESET}: {long_run['verdict']}, {long_run['passed']} of "
      f"{long_run['of']} criteria")
print(f"{measured['seeds']} histories of {measured['years']} years, "
      f"measured {measured['date']}")
print(f"criteria: {long_run['criteria']}\n")


def show(value):
    if isinstance(value, list):
        return " / ".join(show(v) for v in value)
    if value is None:
        return "-"
    return f"{value:g}" if isinstance(value, (int, float)) else str(value)


# pt-v19 to pt-v21's rows carry their words, the real value and the rule.
# pt-v22's carry a band, or the real value where the rule is a ratio, and the
# rule where the grade names one; a row the grade could not read has no
# value. Rows with no words are printed by id.
def verdict(row):
    if row["pass"] is None:
        return "not read"
    return ("pass" if row["pass"] else "FAIL") + (
        "" if row.get("gated", True) else " (reported)")


for row in long_run["rows"]:
    against = (f"real {show(row['real']):>12s}" if "real" in row
               else f"band {show(row.get('band')):>12s}")
    print(f"{row['id']}  {verdict(row)}  {row.get('words', '')}")
    print(f"          model {show(row['value']):>12s}   {against}   "
          f"{row.get('rule', '')}")

pt-v22: pass, 190 of 190 criteria
270 histories of 21 years, measured 2026-11-01
criteria: validation/pt-v21/programme/longrun/CRITERIA.md (adopted 2026-09-23; C4a and C4b added 2026-09-24; C5-C8 registered 2026-09-24); the pt-v22 rows registered in validation/pt-v22/programme/ptv22-registration-23-DRAFT.md (the twenty-third registration)

A1  pass  worst month's volatility in the 2008 and 2020 replays within 30% of real
          model 100.4 / 98.3   real  84.3 / 94.5   within 30%
A2  pass  maximum drawdown in the 2008 and 2020 replays within 30% of real
          model 0.444 / 0.343   real 0.568 / 0.339   within 30%
A3  pass  peak stock correlation in the 2008 and 2020 replays within 0.15 of real
          model 0.813 / 0.864   real 0.748 / 0.872   within 0.15
B1  pass  share of sessions with the VIX above 30
          model        0.068   real        0.082   1/2x to 2x
B2  pass  mean length of a fear spell above VIX 30, sessions
          model           24   real           22   1/2

In plain terms, for the shipped default:

- A 2008 or 2020 replay produces a crash of roughly the right size: its
  worst month's volatility, its maximum drawdown and its peak correlation
  between stocks all land within the tolerance of the real ones.
- Over twenty free-running years the VIX spends 6.8% of sessions above 30,
  against 8.2% in real markets, and a fear spell lasts about as long, 24
  sessions against 22.
- Bear markets of 20% come 1.69 times a decade, against 1.16 in real
  markets. Corrections of 10% come 4.62 times against 2.84, and sessions
  down more than 5% come 7.2 times against 6.2.
- The index returns 7.0% a year over the long run, against a target of
  6.25%. The target is the real index's return with the long rise in
  valuations taken out: a market priced off earnings has no reason to
  reproduce a P/E ratio that drifted up for decades.
- Reading a headline five ticks late earns about 18 basis points, under
  the 20 basis point line where a bot would learn an edge that real markets
  do not offer.
- It passes C4a and C4b, which pt-v19 failed. The lag-1 autocorrelation of
  65-minute returns is -0.044, where pt-v19 read -0.187, and the price-only
  rule furthest over its line on the suite reads 3.0 points below
  buy-and-hold and beats it in 8 markets of 20. On pt-v19 a mean-reversion
  rule that trades every 65 minutes beat buy-and-hold in 18 of 20 by a
  median 13.6 points.
- A value screen on published fundamentals and 12-1 momentum rank the next
  20 sessions about as weakly as they do in real markets (C6, C7), and the
  average cost of size in the book grows with an exponent of 0.46 in the
  order's share of volume, against the 0.5 published studies find (C9).
- Driven by the real 2020-21 macro path, the index falls 38.2% against the
  S&P 500's 33.9% and is back at its high in 168 sessions against 126
  (D2). A timing rule on published macro data gains at most 0.01 points a
  year over buy-and-hold (C10), and trading on a published rate decision
  loses 1.22 points a year (R7b).

A pass is not always a wide one. The next cell shows how much of each
row's tolerance the model uses, the grade's own figure where the record
carries one: 0 means it matches real markets exactly, 1 means it sits on
the edge.


In [7]:
def used(row):
    # The grade's own figure where the record carries one (pt-v22's rows).
    if isinstance(row.get("usage"), (int, float)):
        return row["usage"]
    rule = row.get("rule")
    if rule is None or "real" not in row:
        return None
    pairs = (list(zip(row["value"], row["real"]))
             if isinstance(row["value"], list) else [(row["value"], row["real"])])
    out = []
    for model, real in pairs:
        if not isinstance(model, (int, float)):
            return None
        if rule == "within 30%":
            out.append(abs(model / real - 1) / 0.30)
        elif rule == "within 20%":
            out.append(abs(model / real - 1) / 0.20)
        elif rule == "within 0.15":
            out.append(abs(model - real) / 0.15)
        elif rule.startswith("within 2 points"):
            out.append(abs(model - real) / 2)
        elif rule == "1/2x to 2x":
            out.append(abs(math.log(model / real)) / math.log(2))
        elif rule == "2/3x to 1.5x":
            out.append(math.log(model / real)
                       / math.log(1.5 if model >= real else 2 / 3))
        elif rule == "at most 1":
            # One history in thirty, as a rate: the grade allows n // 30 of n
            # histories, so 9 of the 270 pt-v21's record reads.
            histories = long_run["measured"]["seeds"]
            out.append(model / max(1, histories // 30))
        elif rule == "under 20 bp":
            out.append(model / 20)
        else:
            return None
    return max(out)


scored = [(row["id"], used(row), row.get("words", ""))
          for row in long_run["rows"]]
for rid, share, words in sorted(scored, key=lambda t: -(t[1] or 0)):
    if share is None:
        continue
    flag = "  <- near the edge" if share >= 0.85 else ""
    print(f"  {rid}  {share:4.0%}  {words[:58]}{flag}")

  R6  116%  the driven 2022 market: the market P/E's log change per 10  <- near the edge
  F-bear  110%    <- near the edge
  R4m   94%    <- near the edge
  H5   94%    <- near the edge
  Q4   92%    <- near the edge
  C3   90%  edge from reading a headline 5 ticks late, bp  <- near the edge
  Q2   87%    <- near the edge
  H4   86%    <- near the edge
  DV6   82%  
  CV1   77%  
  SF3:rec_pkg   77%  
  C10   75%  timing rules on published macro data against buy-and-hold,
  C4a   75%  65-minute lag-1 autocorrelation of print returns, median n
  A2   72%  maximum drawdown in the 2008 and 2020 replays within 30% o
  B4   70%  10% corrections per decade
  C10e-cal   68%  
  Q6   65%  
  H3   65%  
  C7   65%  rank IC of 12-1 and 6-1 month momentum against the next 20
  A1   64%  worst month's volatility in the 2008 and 2020 replays with
  Q1   63%  
  DV7sd   62%  
  SF3:curve   61%  
  SF3:rate   61%  
  G7   61%  
  C9   61%  the cost of size in the agent-facing book: exponent and co
 

C2 allows one history in thirty touching the VIX ceiling, so nine of the
270 the record reads, and pt-v22's five use 56% of that. Of the forty
criteria, the nearest its edge is the headline edge, C3: 18 basis points
against a 20 point line, 90% of the tolerance. Then come C10's worst timing
rule and C4a at 75%, the 2008 and 2020 drawdowns (A2) at 72%, and 10%
corrections (B4) at 70%. R6, at 116%, is outside its band: pt-v22 reports
it and doesn't count it. The 2020 replay's worst month has a volatility of
98.3 against a real 94.5, about 4% higher, and 2008's has 100.4 against
84.3, about 19% higher. The 2020 peak correlation between stocks is 0.86
against 0.87.

So a 2008 or 2020 replay is recognisably a crash of about the real size,
and its worst month is a little more violent than the real one. A long
backtest here meets bear markets and corrections more often than history
would give it.

## The gaps that remain

Three measurements sit outside the pass bar: they are reported and
investigated, and they do not stop a preset. They are on the record below.

In [8]:
lever = record["crisis_lever"]
print(f"crisis lever: volatility {lever['vol_at_vix_5']:.1f}% with the VIX "
      f"held at 5, {lever['vol_at_vix_65']:.1f}% held at 65")
print(f"              {lever['ratio']:.2f}x, against {lever['real']:.2f}x "
      f"in real markets\n")

rise = record["structure_rise"]["rows"]["vix_ar1_debiased"]
for days in (252, 504):
    print(f"VIX persistence over {days} days: {rise[f'median_{days}']:.3f}, "
          f"real {facts.REAL_VIX_AR1[days]:.3f}")

crisis lever: volatility 16.9% with the VIX held at 5, 87.5% held at 65
              5.19x, against 6.16x in real markets

VIX persistence over 252 days: 0.956, real 0.930
VIX persistence over 504 days: 0.960, real 0.959


- **The crisis lever is short of real.** Holding the VIX at 65 instead of
  5 makes stocks 5.19 times as volatile here, against 6.16 times in real
  markets. A scenario that forces the VIX high moves prices less than a
  real panic at the same VIX would.
- **Fear lingers longer than real over one year.** Over one year the
  VIX's day-to-day persistence reads 0.956 against a real 0.930, so the
  certification refuses that row, on both of its panels. Over two years it reads 0.960 against 0.959, but
  its rise from one year to two, +0.0017, is below the real +0.0120.
- **Volatility memory is short.** A volatile day predicts tomorrow's
  volatility about as strongly as in real markets, and the effect then
  decays too quickly. The `decay-shape` gap below gives the numbers.

The named gaps each say what they stop you concluding.


In [9]:
for gap in env.GAPS:
    print(f"* {gap.id}: {gap.summary}")
    print(f"    forbids: {gap.forbids}")

* horizon: the certified horizon is 252 days
    forbids: multi-year backtests, and anything keyed on volatility dynamics beyond one year
* decay-shape: volatility memory decays faster than real past the first day
    forbids: strategies whose edge depends on volatility clustering past lag 8: volatility forecasts over more than a week, and vol targeting and risk parity on a one-month or longer estimate
* scenario-magnitude: a driven scenario moves prices at about half to four-fifths of the real size
    forbids: sizing a scenario's impact rather than detecting it
* macro-range: the endogenous macro state cannot reach its own crisis regimes
    forbids: studying inflation regimes or policy crises from the endogenous economy alone
* roster-concentration: a concentrated roster is granted on pt-v19 only, for four sector mixes and the shape rows
    forbids: citing the certification for a concentrated roster on a level or crisis row, past 504 days, on any preset but pt-v19 (pt-v20 and the d

## Checking your own question

`envelope.check` takes a horizon, the statistics a strategy relies on, and
the shape of the roster, and says whether the question is inside the
envelope. Every refusal names the measurement behind it.

In [10]:
questions = [
    ("a one-year momentum study", dict(horizon_days=252,
                                       statistics=["return_acf1"])),
    ("a three-year study",        dict(horizon_days=756,
                                       statistics=["abs_return_acf1"])),
    ("volatility clustering decay", dict(horizon_days=252,
                                         statistics=["abs_return_acf20"])),
    ("a tech-only roster",        dict(horizon_days=252,
                                       sector_concentrated=True)),
    ("sizing a crisis scenario",  dict(horizon_days=252,
                                       scenario_magnitude=True)),
]

import textwrap

for label, kwargs in questions:
    v = env.check(**kwargs)
    print(f"{label:32s} {'INSIDE' if v.inside else 'outside'}")
    if not v.inside:
        for reason in v.reasons:
            print(textwrap.fill(textwrap.shorten(reason, 240), 76,
                                initial_indent="      ",
                                subsequent_indent="      "))

a one-year momentum study        INSIDE
a three-year study               outside
      A 756-day run is longer than the 252 days this model is certified for,
      so the certification does not cover it. At 504 days the model holds
      all 14 shape rows the ruled bands can grade
      (facts.REAL_MARKETS_RULED_504). On the 2015-2025 [...]
      abs_return_acf1 leans on how strongly volatility clusters, and the
      model's clustering is close to real at lag 1 and falls away faster
      than real after it: the |return| autocorrelation reads 0.09479 at lag
      1 against real markets' [...]
volatility clustering decay      outside
      abs_return_acf20 depends on the decay shape, which is a mechanism gap:
      the |return| autocorrelation reads 0.09479 at lag 1 against real
      markets' 0.1071 and falls away faster than real after it (0.00115
      against 0.0286 at lag 20). It is [...]
a tech-only roster               outside
      the roster is sector-concentrated and its mix 

The three-year refusal grades the two-year panel on both rulers, as the
cell above did: every row in on the long-record bands, and
`excess_kurtosis` out on the 2015-2025 ones. It refuses on the horizon.

`check` reads the one-year envelope, so it refuses a three-year study even
though the long-run check passes all 190 rows that count toward the
grade. The two
answer different questions. The
long-run check says that crash sizes, fear, bear markets and long-run
return look like a real market's over twenty years. The envelope says
which detailed statistics are certified, and it certifies them at one year
and no further. A multi-year backtest is on firmer ground than it was
before 0.8.0. It is still outside what `check` certifies, so test it
across seeds rather than trusting one run.

## Choosing a preset

`pt-v22` is the default from 0.11.0. `pt-v21` was the default in 0.10.x,
`pt-v20` from 0.8.5 to 0.9.1, `pt-v19` in 0.8.0 and 0.8.1, and `pt-v18` through 0.7.x. Every earlier preset stays selectable, so work published against
one of them keeps reproducing. Each shipped default carries a measured
record, and `envelope.regressions` names the one-year rows a preset gives
up against the shipped one, pt-v22.

In [11]:
defaults = ("pt-v3", "pt-v10", "pt-v12", "pt-v14", "pt-v16", "pt-v18", "pt-v19",
            "pt-v20", "pt-v21")
print(f"{'preset':8s} {'default':>8s}  {'crisis lever':>12s}  {'long run':>9s}  "
      f"gives up against {env.PRESET} at one year")
for name in defaults:
    r = tf.preset_record(name)
    lost = env.regressions({k: v for k, v in r["panel_252"].items()
                            if v is not None})
    lr = r.get("long_run")
    verdict = f"{lr['passed']}/{lr['of']}" if lr else "not run"
    print(f"{name:8s} {r['default_since']:>8s}  {r['crisis_lever']['ratio']:11.2f}x"
          f"  {verdict:>9s}  {', '.join(lost) or 'nothing'}")

preset    default  crisis lever   long run  gives up against pt-v22 at one year
pt-v3       0.1.0         3.08x    not run  sector_excess_corr, volume_change_acf1
pt-v10      0.2.0         5.04x    not run  volume_change_acf1
pt-v12      0.3.0         5.95x    not run  nothing
pt-v14      0.4.0         6.19x    not run  nothing
pt-v16      0.6.0         6.50x    not run  nothing
pt-v18      0.7.0         7.06x    not run  nothing
pt-v19      0.8.0         5.22x      15/17  nothing
pt-v20      0.8.5         5.11x      40/40  nothing
pt-v21     0.10.0         4.98x      40/40  nothing


On the one-year panel the defaults from `pt-v12` on are hard to tell
apart: none gives up a row against pt-v22. `pt-v3` gives up sector
co-movement and volume-change autocorrelation, and `pt-v10` the second of
those. What separates the recent defaults is the long-run check, which is
on record for them alone. `pt-v22` passes all 190 rows that count toward
its grade, read on 270 histories. `pt-v21` and `pt-v20` passed all forty
criteria on their own grades. `pt-v19`'s record carries the seventeen it
was graded on and fails two, the price-only rows; graded on all forty
beside pt-v20, it fails sixteen. Their crisis levers are close: 5.19x for
pt-v22, 4.98x for pt-v21, 5.11x for pt-v20 and 5.22x for pt-v19, all below
real.

**Take the default, `pt-v22`, unless you are reproducing work published
against an older preset. For a study longer than a year, read the long-run
check above, run several seeds, and remember that `check` does not
certify it.**

## Summary

- Realism is a set of measurements against real-market bands, with the
  failures named, and no single score.
- At one year all 19 rows are in band on the long-record bands, with the
  index's return and its rate of 3% falls read over 360 seeds. On the
  2015-2025 bands every row is in. At two years one row, the fat tails of
  daily returns, is under the 2015-2025 floor.
- `pt-v22` passes all 190 long-run rows that count toward its grade,
  including the two price-only rows `pt-v19` fails. Of the forty criteria,
  the nearest the edge are the headline edge, a timing rule on published
  macro data and the 65-minute reversal, and a 2008 replay's worst month
  is about 19% more volatile than the real one.
- The crisis lever, the VIX's persistence over one year and its rise to
  two years, and volatility memory are still off, and are on the record.
- Good results here do not predict real returns.

Full documentation: <https://docs.tradefloor.dev>